# Частина 2 — Агент діє

У частині 1 агент навчився **бачити світ**: прочитав резюме, забрав вакансії з DOU, структурував їх, відсіяв зайве й оцінив відповідність — на виході впорядкований `shortlist` із причинами та `gap` (чого бракує).

Тепер дамо йому **руки**. План частини 2:

1. **Recap** — піднімаємо стан частини 1 зі снапшоту.
2. **Анатомія агента** — цикл `стан → LLM вирішує → інструмент → новий стан`; tool use на іграшковому прикладі.
3. **Лист** — `draft_application`: мотивація + питання по `gap` у кінці.
4. **Approval gate** — агент пропонує, людина підтверджує; лише потім реальна дія.
5. **Відгук на DOU** + читання відповіді (Gmail/IMAP).
6. **Календар** — розбір часу зустрічі (LLM) + детермінований добір слота (без LLM).

> Наскрізний принцип: **руки окремо, голова окремо**. Конектори — німий I/O, рішення приймає цикл агента, дозвіл дає людина.

## Підготовка середовища

Той самий каркас, що в частині 1: додаємо корінь проєкту в `sys.path` і підтягуємо секрети з `.env` (тепер знадобляться ще `AGENT_EMAIL`, `GMAIL_APP_PASSWORD`, `DOU_SESSIONID`, `DOU_CSRFTOKEN`).

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # корінь проєкту (батьківська до notebooks/)

from dotenv import load_dotenv
load_dotenv()  # GOOGLE_API_KEY, MISTRAL_API_KEY, AGENT_EMAIL, GMAIL_APP_PASSWORD, DOU_*

print("Оточення завантажено")

## Recap: піднімаємо стан частини 1

Частина 1 зберегла стан у `data/state.json` (просто `model_dump`, не БД). Завантажуємо його й **відновлюємо ті самі Pydantic-моделі** — далі працюємо з типізованими об'єктами, ніби нічого й не переривалось.

Якщо снапшоту немає (його не зберігали або це чистий запуск) — повідомляємо й просимо пройти частину 1. На вебінарі це наш фолбек: за потреби переганяємо пайплайн ч.1 наживо.

In [ ]:
import json, pathlib
from core.models import CandidateProfile, SearchPreferences, Vacancy, MatchResult

state_path = pathlib.Path("../data/state.json")

if state_path.exists():
    state = json.loads(state_path.read_text(encoding="utf-8"))
    profile = CandidateProfile(**state["profile"])
    prefs = SearchPreferences(**state["prefs"])
    # відновлюємо shortlist: (Vacancy, MatchResult, missing) — як наприкінці ч.1
    shortlist = [
        (Vacancy(**it["vacancy"]), MatchResult(**it["match"]), it["missing"])
        for it in state["shortlist"]
    ]
    print(f"\u2705 стан завантажено: {profile.full_name}, у шортлисті {len(shortlist)} вакансій")
    for v, m, gaps in shortlist[:5]:
        flag = f"  \u26a0 уточнити: {gaps}" if gaps else ""
        print(f"  {m.score:>3}  {(v.company or '\u2014')[:20]:20} — {(v.role or '\u2014')[:30]}{flag}")
else:
    print("\u26a0\ufe0f data/state.json не знайдено — пройди частину 1 (part1_agent_sees.ipynb) до кінця.")

## Лист від AI-агента: мотивація + питання по gap (Task 11)

Перша дія агента — написати лист на топову вакансію зі шортлиста.

## Фішка: лист пише сам AI-агент

Ключова ідея вебінару: лист роботодавцю надсилає **не кандидат, а його персональний AI-агент** — і це навмисно видно з тексту. Сам факт, що в людини є такий агент, — **сигнал її технічної кмітливості** (саме того, чого ми тут і вчимося). Тому тон листа — яскравий, живий, чіпкий, а не шаблонний «мотиваційний лист №5».

## Мова листа = мова оголошення

У Task 6 ми додали в `Vacancy` поле `posting_language` (мова, якою написане оголошення). Лист пишемо **тією ж мовою**: укр вакансія → укр лист, англ → англ. Це елементарна повага й вищий шанс на відповідь.

## Розподіл відповідальності

- **LLM** — *текст* (мотивація + питання + потрібна мова): його сильна сторона.
- **Код** — *кому* (`to = apply_url`) і *чи слати взагалі*: це не робота LLM. `to` проставляємо самі, відправку гейтить людина (наступний крок).

Згадаймо рішення з ч.1: окремої гілки «уточнюючий лист» немає — `gap` просто підмішує блок питань у кінець. Результат — типізований `EmailDraft`, який людина бачить до відправки.


In [ ]:
import os
from connectors.llm import draft_application

profile_summary = f"{profile.full_name}, {profile.title}, {profile.seniority}, стек: {profile.tech_stack}"

# беремо саме нашу вакансію (exsol) зі шортлиста
vac, match, gaps = next(t for t in shortlist if "363796" in (t[0].apply_url or ""))
print(f"Готуємо лист на: {vac.company} — {vac.role} "
      f"(score {match.score}, мова {vac.posting_language}, gap={gaps})\n")

draft = draft_application(
    vac.raw_text, profile_summary,
    sender=os.environ.get("AGENT_EMAIL", "candidate@example.com"),
    missing=gaps,
    language=vac.posting_language,        # лист мовою оголошення
    candidate_name=profile.full_name,     # імʼя з резюме — ДОСЛІВНО, без транслітерації
)
draft.to = vac.apply_url or ""            # кому слати — рішення коду, не LLM

print(f"[{draft.kind}] до: {draft.to}")
print(f"Тема: {draft.subject}\n")
print(draft.body)

## Відгук на DOU + approval gate (Task 10 + 12)

Лист готовий. Тепер — дія в світі: відгук через форму DOU. Тут три принципи.

### Руки окремо, голова окремо
`DouFormSender` (реалізує порт `MessageSender`) — **німий I/O**: лише викликає хелпер `dou_helper.apply_for_vacancy`, який робить POST на форму DOU (поля `descr`, `csrfmiddlewaretoken`, `user_cv`; куки `sessionid`/`csrftoken`). Конектор нічого не вирішує. Низькорівневий HTTP схований у `helpers/` — на вебінарі показуємо оглядово.

### Approval gate — людина між наміром і дією
Агент *пропонує* — людина *вирішує*. Між текстом від LLM і реальним відправленням завжди стоїть людина (`input()`). Це не обмеження демо, а **принцип проєктування**: автономні дії без нагляду — найбільший ризик агентів.

### dry_run — безпечний дефолт
`DouFormSender(dry_run=True)` нічого реально не шле — лише друкує, що відправив би. Реальний відгук вмикається одним прапорцем `dry_run=False` (коли є активна вакансія й чинні куки). Тому код можна ганяти скільки завгодно без сайд-ефектів.

> Структуру форми відгуку (`POST` на URL вакансії, поля `descr`/`user_cv`/csrf) ми розвідали на реальній internal-вакансії DOU. Не всі вакансії такі: «external» ведуть на сайт компанії — для них ця форма не діє.


In [ ]:
from connectors.dou_form import DouFormSender
from core.models import Attachment

# dry_run=True — МОК (нічого не шлемо). Реальний відгук: DouFormSender(dry_run=False)
sender = DouFormSender(dry_run=True)

# резюме як вкладення (піде в поле user_cv форми DOU)
resume = Attachment(filename="resume.pdf",
                    path="../data/Kostiantyn_Zivenko_Resume_EN.pdf",
                    mime_type="application/pdf")

# показуємо чернетку людині
print(f"[{draft.kind}] до: {draft.to}")
print(f"Тема: {draft.subject}\n")
print(draft.body)
print("\n" + "─" * 50)

# APPROVAL GATE: реальна дія лише після явного підтвердження людини
if input("\nВідправити відгук на DOU? [y/N] ").strip().lower() == "y":
    sender.send(draft.to, draft.subject, draft.body, attachments=[resume])
    print("→ далі очікуємо відповіді роботодавця на email (Gmail/IMAP)")
else:
    print("✋ скасовано")


## Память агента: стан розмови

Відгук надіслано — далі почнеться **листування**, а воно **не однокрокове**: агент пропонує час → роботодавець відповідає → агент реагує. Якщо агент нічого не памʼятає, голе «так, підходить» втрачає сенс (на що саме «так»?).

Тому заводимо `Conversation` — память про діалог по одній вакансії: на якій ми **стадії** (`applied → negotiating → scheduled` / `rejected`) і який час ми **останнім запропонували** (`last_proposed_slot`). Завдяки цьому згода без повторення часу читається правильно.

> Саме цей стан фреймворки (LangGraph / LangChain) дають «з коробки». Ми ж тримаємо його **явно й просто**, окремим обʼєктом — щоб бачити механіку, перш ніж ховати її за фреймворком.

In [ ]:
from core.models import Conversation

# память про діалог саме по нашій вакансії (стартова стадія = applied)
conv = Conversation(company=vac.company)
print(f"🧠 Розмову створено: {conv.company} | стадія: {conv.stage.value}")

## Читання відповіді: ImapReceiver (Task 13)

Відгук пішов (реально або dry-run). Тепер агент має **почути відповідь** — це друга «рука»: читання вхідних листів.

### ImapReceiver — німий I/O на отримання
`ImapReceiver` реалізує порт `MessageReceiver`: підключається до Gmail агента через IMAP, дістає непрочитані (`UNSEEN`) листи й повертає типізовані `IncomingMessage` (`sender`/`subject`/`body`). Як і відправка — лише I/O, жодних рішень.

### Демо-стійкість: реальний шлях + мок-фолбек
- **Реальний шлях** (краще для демо): надсилаємо листа з будь-якої пошти на `AGENT_EMAIL` — `ImapReceiver` його читає. Жива петля «роботодавець відповів → агент прочитав».
- **Фолбек** `MockReceiver`: заготовлена відповідь без мережі — якщо скринька порожня або IMAP недоступний на ефірі.

Обидва реалізують той самий порт, тож перемикання — один рядок. Це і є сенс ports & adapters: канал міняється, ядро — ні.

> Заготовлена відповідь навмисно містить і вилку (закриває `gap`), і пропозицію часу зустрічі — наскрізний місток до фінальної гілки «співбесіда».


In [ ]:
from connectors.email import ImapReceiver, MockReceiver

# Реальний шлях: лист від "роботодавця" на AGENT_EMAIL (можна надіслати собі самому).
receiver = ImapReceiver()
inbox = receiver.fetch_unread(limit=5)

# Фолбек на демо: якщо непрочитаних немає — беремо заготовлену відповідь
if not inbox:
    print("📭 непрочитаних немає — використовуємо мок-відповідь для демонстрації\n")
    inbox = MockReceiver().fetch_unread()

reply = inbox[0]          # reply: IncomingMessage
print(f"Від:  {reply.sender}")
print(f"Тема: {reply.subject}\n")
print(reply.body)


## Обробка відповіді: матчинг + розвилка (Task 14)

Агент отримав лист (Task 13). Тепер reasoning: **зрозуміти лист і вирішити, що робити**.

### Крок 1: один розбір — `analyze_reply`
LLM розбирає лист у `ReplyAnalysis`: до якої вакансії він (`matched_company`), чи це відмова, які дані дослали (вилка/формат/локація), чи пропонують час зустрічі. Один виклик — уся потрібна структура.

### Крок 2: матчинг — без LLM
`match_reply_to_vacancy` зіставляє згадку компанії з нашим шортлистом — звичайне порівняння рядків. LLM витягнув назву, код зіставив (знову: що детерміноване — без LLM).

### Крок 3: routing за **набором** чинників
Чинники не взаємовиключні — діємо за тим, що в листі:
- **відмова** → стоп, наступна вакансія;
- **дослані дані** → дозаповнюємо вакансію (`apply_reply_updates`, лише порожні поля) → `gap` зменшується → **переоцінюємо** (`score_match`);
- **пропозиція часу** → гілка співбесіди (Task 16).

Так замикається петля з частиною 1: ті самі чисті функції (`missing_fields`, `score_match`) складаються заново на нових даних.

> Зараз показуємо гілку **дозаповнення** (A); «зустріч» (Task 16) і «відмова» — це той самий routing, лише інші дії.


In [ ]:
from connectors.llm import analyze_reply, score_match
from core.logic import match_reply_to_vacancy, apply_reply_updates, missing_fields

vacancies = [v for v, _, _ in shortlist]
conv.history.append(reply)        # память розмови: фіксуємо вхідний лист
profile_summary = f"{profile.title}, {profile.seniority}, стек: {profile.tech_stack}"
prefs_summary = f"ролі: {prefs.desired_roles}, від {prefs.min_salary}"

# 1) розбір листа — даємо ПОВНИЙ лист (назва компанії часто у From/Subject, не в тілі)
reply_text = f"Від: {reply.sender}\nТема: {reply.subject}\n\n{reply.body}"
analysis = analyze_reply(reply_text, [v.company for v in vacancies])
print(f"Розбір: компанія={analysis.matched_company!r}, відмова={analysis.is_rejection}, "
      f"час={analysis.meeting_proposal!r}")
print(f"Дослані: salary={analysis.salary_min}-{analysis.salary_max}, "
      f"format={analysis.work_format}, location={analysis.location}\n")

# 2) матчинг до вакансії (без LLM)
vac = match_reply_to_vacancy(analysis.matched_company, vacancies)

# 3) routing за набором чинників
if analysis.is_rejection:
    print("❌ Відмова — переходимо до наступної вакансії зі шортлиста.")
elif vac is None:
    print("⚠️ Не вдалося зіставити лист із вакансією зі шортлиста.")
else:
    # (A) дозаповнення, якщо роботодавець дослав дані
    if any([analysis.salary_min, analysis.salary_max, analysis.work_format, analysis.location]):
        before = missing_fields(vac, prefs)
        apply_reply_updates(vac, analysis)          # дозаповнюємо лише порожні поля
        after = missing_fields(vac, prefs)
        print(f"✏️  Дозаповнено «{vac.company}»: gap {before} → {after}")
        m = score_match(vac.raw_text + "\n" + reply.body, profile_summary, prefs_summary)
        print(f"   Переоцінка: score {m.score}")
    # (B) зустріч, якщо запропонували час
    if analysis.meeting_proposal:
        print(f"📅 Пропозиція зустрічі: {analysis.meeting_proposal!r} → гілка співбесіди (Task 16).")

## Календар: добір слота без LLM (Task 15)

Відповідь містить пропозицію часу (Task 14 це показав). Перш ніж відповідати — агенту потрібен **календар пошукача** і логіка добору слота.

### Знову: розбір — LLM, рішення — код
Це серце тези про конектори:
- **розбір дати** з листа («четвер о 15:00» → `datetime`) — робота LLM (наступний крок);
- **рішення «вільно / зайнято / найближчий слот»** — ДЕТЕРМІНОВАНА логіка → чистий Python у `core/calendar.py`.

### LocalCalendar (варіант A)
Список зайнятих інтервалів + робочі години. Без OAuth, без мережі — надійно й тестовано. `resolve_slot(proposed, cal)` повертає `SlotDecision`:
- `accept` — час вільний;
- `propose_alternative` — зайнято/поза годинами → найближчий вільний слот.

Рішення повертаємо **як дані** (`SlotDecision`), а не діємо одразу — щоб далі вмонтувати approval gate.

> Реальний Google Calendar (камео, Task 17) реалізує той самий контракт `CalendarProvider` — локальний і хмарний календарі взаємозамінні (ports & adapters).


In [ ]:
from datetime import datetime
from core.calendar import LocalCalendar
from core.logic import resolve_slot

# календар пошукача: кілька зайнятих інтервалів (робочі години 10:00–19:00)
cal = LocalCalendar(busy=[
    (datetime(2026, 7, 2, 11), datetime(2026, 7, 2, 12)),
    (datetime(2026, 7, 2, 15), datetime(2026, 7, 2, 16)),
])

# приклади рішень (детерміновано, без LLM)
for label, dt in [("вільний 14:00", datetime(2026, 7, 2, 14)),
                  ("зайнятий 15:00", datetime(2026, 7, 2, 15)),
                  ("вечір 21:00",    datetime(2026, 7, 2, 21))]:
    d = resolve_slot(dt, cal)
    print(f"{label:16} → {d.action:20} {d.slot:%Y-%m-%d %H:%M}")


## Повний цикл: співбесіда (Task 16)

Тепер з'єднуємо все в один прохід — від листа з пропозицією часу до події в календарі й відповіді роботодавцю.

### Три інструменти складаються в один прохід
1. **Розбір часу** (LLM): `parse_meeting` → `InterviewProposal{when}`. Відносна дата («четвер о 15:00») потребує орієнтиру — передаємо поточну дату.
2. **Рішення** (без LLM): `resolve_slot` → `accept` / `propose_alternative`; за accept — бронюємо слот (`cal.book`).
3. **Лист-відповідь** (LLM, від AI-агента): `draft_meeting_reply` — підтвердження або контр-пропозиція, мовою вакансії.

### Хто що робить
Розгалуження роблять **правила** (`resolve_slot`), текст — **LLM**, відправку — **людина** (approval gate), реальну подію — **календар**. Видно, як чисті блоки складаються в агентний прохід.

> Відправка через `GmailSender(dry_run=True)` — мок. Для реального листа роботодавцю: `dry_run=False` (SMTP на `AGENT_EMAIL`).


In [ ]:
import os
from datetime import datetime, timedelta
from connectors.llm import parse_meeting, draft_meeting_reply
from connectors.email import GmailSender
from core.calendar import LocalCalendar
from core.logic import resolve_slot, resolve_meeting_time
from core.models import ConversationStage

# РЕАЛЬНИЙ календар із запасним варіантом — той самий контракт CalendarProvider
try:
    from connectors.gcal import GoogleCalendar
    cal = GoogleCalendar()                       # читає/пише твій Google Calendar
    print("📅 реальний Google Calendar")
except Exception as e:
    cal = LocalCalendar(busy=[(datetime(2026, 7, 2, 11), datetime(2026, 7, 2, 12)),
                              (datetime(2026, 7, 2, 15), datetime(2026, 7, 2, 16))])
    print(f"📅 запасний LocalCalendar ({type(e).__name__})")

# 1) LLM витягує час, ЯКЩО він є у листі (інакше when=None — голе «так»)
proposal = parse_meeting(analysis.meeting_proposal, datetime.now().strftime("%Y-%m-%d (%A)"))

# 2) ПАМʼЯТЬ розмови: голе «так» → беремо НАШ останній запропонований слот
when = resolve_meeting_time(proposal.when, conv)

if when is None:
    print("🤔 Часу не зрозуміли й нема що згадати — лишаємось на стадії узгодження.")
else:
    print("Час до розгляду:", f"{when:%Y-%m-%d %H:%M}")
    decision = resolve_slot(when, cal)
    print("Рішення:", decision.action, "→", f"{decision.slot:%Y-%m-%d %H:%M}")

    # 3) оновлюємо СТАН розмови за рішенням
    if decision.action == "accept":
        link = cal.create_event(f"Співбесіда: {conv.company}",
                                decision.slot, decision.slot + cal.slot)
        conv.agreed_slot = decision.slot
        conv.stage = ConversationStage.scheduled
        print("📌 подію створено:", link)
    else:
        conv.last_proposed_slot = decision.slot      # запамʼятовуємо, що ми пропонуємо
        conv.stage = ConversationStage.negotiating

    # 4) лист-відповідь від AI-агента, мовою вакансії
    draft = draft_meeting_reply(
        decision.action, decision.slot,
        employer=reply.sender, sender=os.environ["AGENT_EMAIL"],
        language=vac.posting_language if vac else None,
        candidate_name=profile.full_name,     # імʼя з резюме — ДОСЛІВНО
    )
    draft.to = reply.sender   # відповідаємо на email роботодавця
    print(f"\n[{draft.kind}] до: {draft.to}\nТема: {draft.subject}\n\n{draft.body}")

    # 5) approval gate + відправка
    gmail = GmailSender(dry_run=False)
    if input("\nНадіслати відповідь роботодавцю? [y/N] ").strip().lower() == "y":
        gmail.send(draft.to, draft.subject, draft.body)
        print(f"✅ надіслано | стан розмови: {conv.stage.value}")
    else:
        print("✋ скасовано")

## Камео: реальний Google Calendar (Task 17)

У Task 16 ми вже підключили **реальний** календар: `cal = GoogleCalendar()` (з фолбеком). Тут — підсумок, чому це було так легко.

### Один інтерфейс, різні адаптери
`LocalCalendar` і `GoogleCalendar` (`connectors/gcal.py`) реалізують один порт `CalendarProvider` (`is_available` / `next_free_slot` / `create_event`). `GoogleCalendar` робить те саме через Google Calendar API (`freebusy` + `events.insert`). Тому `resolve_slot` і цикл агента **не змінились** — помінявся лише адаптер. Це той самий аргумент, що й MCP: логіку пишемо раз, конектор лише реалізує контракт.

### Setup (один раз)
1. Google Cloud Console → проект → увімкнути **Google Calendar API**.
2. Google Auth Platform → Audience: **External**, додати себе в **Test users**.
3. Credentials → OAuth client ID → **Desktop app** → завантажити `credentials.json` у корінь.
4. Перший запуск `GoogleCalendar()` відкриє браузер для згоди → збережеться `token.json` (далі без браузера).

> Якщо `credentials.json` немає — Task 16 робить фолбек на `LocalCalendar`, тож демо не зривається.


In [ ]:
# Task 17 — підсумок: той самий resolve_slot уже працював на cal із Task 16.
# Якщо cal — GoogleCalendar, ми щойно читали РЕАЛЬНІ зайняті слоти (freebusy)
# і за accept створили реальну подію. Заміна LocalCalendar ↔ GoogleCalendar
# не торкнулась ні resolve_slot, ні циклу агента — лише адаптера.
from datetime import datetime
from core.logic import resolve_slot

probe = datetime(2026, 7, 3, 14)
print(f"Поточний адаптер: {type(cal).__name__}")
print(f"resolve_slot({probe:%Y-%m-%d %H:%M}) → {resolve_slot(probe, cal).action}")
print("Той самий код, інший адаптер — у цьому й сила портів (і MCP).")


# Фінал: що ми побудували, MCP і межі

## Повний агент end-to-end

За дві зустрічі ми зібрали агента, що проходить весь шлях:

```
Частина 1 (агент бачить):
  резюме (PDF) → CandidateProfile
  побажання (руками) → SearchPreferences
  вакансії DOU → list[Vacancy]
  фільтр (без LLM) → повнота даних (без LLM) → матчинг (LLM) → shortlist

Частина 2 (агент діє):
  лист від AI-агента (мовою вакансії) → approval gate → відгук на DOU
  читання відповіді (IMAP) → розбір + матчинг + routing
    ├─ дослали дані → дозаповнення → переоцінка
    ├─ пропозиція часу → календар → лист-підтвердження/контр-пропозиція
    └─ відмова → наступна вакансія
```

## Принципи, які проходять наскрізь

| Принцип | Де проявився |
|---|---|
| **Structured output** | один `extract()` через instructor: текст → Pydantic |
| **Модель за задачею** | легка для парсингу, reasoning-модель для суджень |
| **Крос-провайдерний фолбек** | Gemini → Mistral в одному ланцюзі |
| **Алгоритм, де факт; LLM, де судження** | фільтр і повнота — код; матчинг і листи — LLM |
| **Ports & adapters** | джерело/пошта/календар — змінні адаптери, ядро незмінне |
| **Human-in-the-loop** | approval gate перед кожною реальною дією |

## Зв'язок з MCP

Усе, що ми зробили руками — `DouFormSender`, `ImapReceiver`, `GmailSender`, `GoogleCalendar` — це **інструменти (tools)**, які стандартизує **MCP** (Model Context Protocol). Наступний крок розвитку: замінити власні конектори на готові MCP-сервери (пошта, календар, CRM, кілька джерел вакансій), лишивши **незмінними `core/` і цикл агента**. Те, що в нас порт + адаптер — у MCP контракт + сервер. Ідея та сама, лише стандартизована.

## Межі та етика (важливо)

- **Вартість.** Кожен крок reasoning — це токени й гроші. Тому ми робили максимум детерміновано (фільтр, повнота, календар) і берегли LLM на судження.
- **Надійність.** LLM стохастична; на free tier ще й часто недоступна. Звідси — фолбеки, валідація схемою, ретраї. Але 100% гарантії немає — критичні рішення лишаємо людині.
- **Етика й спам.** Масові автоматичні відгуки — це ризик зловживання й шуму для роботодавців. Наша відповідь: **approval gate** (людина підтверджує кожну дію) і **прозорість** (лист чесно каже, що його склав AI-агент). Автоматизація має підсилювати людину, а не імітувати її потай.

## Куди рости

Кілька джерел вакансій (новий `VacancySource`), реальні MCP-сервери замість конекторів, пам'ять між запусками (БД замість `state.json`), складніший reasoning-цикл із кількома інструментами. Каркас для цього вже є — `core/` і порти не доведеться переписувати.
